### QPDK Coupled Resonator — Driven Simulation

This notebook builds a compact coupled resonator from QPDK cells, converts etch
layers to conductor geometry, and runs a Palace driven simulation to extract
S-parameters around the resonance near 7.5 GHz.

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook demonstrates using the `gsim.palace` API to run a driven simulation on a microstrip transmission line with via ports.

**Requirements:**

- IHP PDK: `uv pip install qpdk`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

In [ ]:
import gdsfactory as gf

from qpdk import PDK, cells
from qpdk.cells.airbridge import cpw_with_airbridges
from qpdk.tech import LAYER, route_bundle_sbend_cpw

PDK.activate()


@gf.cell
def resonator_compact(coupling_gap: float = 20.0) -> gf.Component:
    """Compact coupled resonator with S-bend CPW routes."""
    c = gf.Component()

    res = c << cells.resonator_coupled(
        coupling_straight_length=300, coupling_gap=coupling_gap
    )
    res.movex(-res.size_info.width / 4)

    left = c << cells.straight()
    right = c << cells.straight()

    w = res.size_info.width + 100
    left.move((-w, 0))
    right.move((w, 0))

    route_bundle_sbend_cpw(
        c,
        [left["o2"], right["o1"]],
        [res["coupling_o1"], res["coupling_o2"]],
        cross_section=cpw_with_airbridges(
            airbridge_spacing=250.0, airbridge_padding=20.0
        ),
    )

    c.kdb_cell.shapes(LAYER.SIM_AREA).insert(c.bbox().enlarged(0, 100))

    c.add_port(name="o1", port=left["o1"])
    c.add_port(name="o2", port=right["o2"])
    return c


component = resonator_compact(coupling_gap=15.0)
_c = component.copy()
_c.draw_ports()
_c

### Convert QPDK etch layers to conductor geometry

In [ ]:
from gsim.common.stack.qpdk_utils import create_etched_component

etched = create_etched_component(component)
etched

### Configure simulation

In [ ]:
from gsim.palace import DrivenSim

sim = DrivenSim()
sim.set_geometry(etched)
sim.set_stack(substrate_thickness=500, air_above=500)
sim.add_cpw_port(
    "o1", layer="SUPERCONDUCTOR", s_width=10.0, gap_width=6.0, length=5.0, offset=2.5
)
sim.add_cpw_port(
    "o2", layer="SUPERCONDUCTOR", s_width=10.0, gap_width=6.0, length=5.0, offset=2.5
)
sim.set_driven(fmin=7.75e9, fmax=7.8e9, num_points=100)

In [ ]:
sim.set_output_dir("./sim_qpdk_resonator")
sim.mesh(preset="graded", margin=0)
sim.plot_mesh(
    show_groups=["superconductor", "P", "sapphire", "vacuum"], interactive=False
)

In [ ]:
sim.write_config()
results = sim.run()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(results["port-S.csv"])
df.columns = df.columns.str.strip()

freq = df["f (GHz)"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6))

# Magnitude plot
ax1.plot(freq, df["|S[1][1]| (dB)"], marker=".", label="S11")
ax1.plot(freq, df["|S[2][1]| (dB)"], marker=".", label="S21")
ax1.set_xlabel("Frequency (GHz)")
ax1.set_ylabel("Magnitude (dB)")
ax1.set_title("S-Parameters — QPDK Coupled Resonator")
ax1.legend()
ax1.grid(True)

# Phase plot
ax2.plot(freq, df["arg(S[1][1]) (deg.)"], marker=".", label="S11")
ax2.plot(freq, df["arg(S[2][1]) (deg.)"], marker=".", label="S21")
ax2.set_xlabel("Frequency (GHz)")
ax2.set_ylabel("Phase (deg)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()